# 05 — XGBoost Global Demand Model: M5 Walmart Demand Intelligence

**Goal:** Train a single global XGBoost model across all 30,490 product-store
series. Demonstrate that price, SNAP, and lag features close the univariate
signal ceiling — the three months (Apr 2015, May 2015, Jan 2016) where both
SARIMA and Prophet failed regardless of parameterization.

**Inputs:** `features_train.parquet`, `features_val.parquet`, `feature_cols.pkl`

**Outputs:** `xgb_model.pkl`, `xgb_quantile_models.pkl`, `xgb_predictions_val.parquet`, `xgb_cv_results.csv`

**Evaluation:** Three expanding walk-forward folds. Optuna tunes on Folds 1–2.
Fold 2 params frozen before Fold 3 runs. Fold 3 touched exactly once.

| What | Metric | Which folds |
|---|---|---|
| Optuna objective + early stopping | log-RMSE | All folds |
| Fold quality reporting | log-RMSE, MAE, bias (non-zero rows) | All folds |
| Signal ceiling vs SARIMA/Prophet | MAPE on representative series, monthly | Fold 3 only |
| Quantile calibration | Pinball loss + empirical coverage | Fold 3 only |

> **Demand proxy reminder:** Observed sales proxy true latent demand. Zero sales
> may reflect a stockout or genuine absence — indistinguishable without inventory
> data. All outputs are demand approximations, not exact demand recovery.

## 1. Imports and Setup

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import pickle
import os
import time
import warnings
import subprocess

from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams['figure.figsize'] = (14, 5)
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
PROCESSED_DIR = '../data/processed'
REP_SERIES    = 'FOODS_3_163_CA_3_validation'

# ── Walk-forward fold boundaries ───────────────────────────────────────────
# Monitor set = last 60 days of each training window, carved out before fit.
# Evaluation window = the 12 months immediately after.
FOLDS = {
    'fold_1': {
        'train_start':   '2011-02-01',
        'monitor_start': '2012-12-02',  # last 60 days of fold 1 training
        'train_end':     '2013-01-31',
        'val_start':     '2013-02-01',
        'val_end':       '2014-01-31',
    },
    'fold_2': {
        'train_start':   '2011-02-01',
        'monitor_start': '2013-12-02',  # last 60 days of fold 2 training
        'train_end':     '2014-01-31',
        'val_start':     '2014-02-01',
        'val_end':       '2015-01-31',
    },
    'fold_3': {
        'train_start':   '2011-02-01',
        'monitor_start': '2014-12-02',  # last 60 days of fold 3 training
        'train_end':     '2015-01-31',
        'val_start':     '2015-02-01',
        'val_end':       '2016-01-31',
    },
}

# ── XGBoost constants ──────────────────────────────────────────────────────
EARLY_STOPPING_ROUNDS = 50
OPTUNA_TRIALS         = 50
TARGET_COL            = 'target'

# Default parameters — untuned floor that Optuna must beat.
# Standard starting point: depth 6, moderate learning rate,
# mild subsampling, no regularization beyond L2=1.
DEFAULT_PARAMS = {
    'max_depth':          6,
    'learning_rate':      0.1,
    'subsample':          0.8,
    'colsample_bytree':   0.8,
    'min_child_weight':   1,
    'reg_alpha':          0.0,
    'reg_lambda':         1.0,
    'objective':          'reg:squarederror',
    'tree_method':        'hist',
    'device':             'cuda',
    'random_state':       42,
    'n_estimators':       2000,
}

# Frozen best params from Fold 2 Optuna — populated in Section 6, used in Section 7.
# Written here as a named constant so the freeze is visible and auditable.
# DO NOT modify after Section 6 is complete.
BEST_PARAMS = None  # replaced with dict after tuning

# ── Tier 1 / Tier 2 evaluation ─────────────────────────────────────────────
def eval_log_scale(y_true_log, y_pred_log, label):
    """
    Tier 1 / Tier 2 metric. Operates in log1p space on non-zero actual rows.
    
    Non-zero filter: rows where actual=0 are structural gap rows (product
    unavailable, not absent demand). Including them inflates error metrics
    without measuring forecast quality. Filter applied and row count reported
    explicitly so the restriction is always visible.
    
    Bias is reported alongside accuracy — a model that is consistently 10%
    low causes systematic stockouts in a way that random ±20% error does not.
    """
    mask = y_true_log > 0
    n    = mask.sum()

    rmse = np.sqrt(mean_squared_error(y_true_log[mask], y_pred_log[mask]))
    mae  = mean_absolute_error(y_true_log[mask], y_pred_log[mask])
    bias = float(np.mean(y_pred_log[mask] - y_true_log[mask]))

    print(f'{label}  [non-zero rows: {n:,}]')
    print(f'  log-RMSE: {rmse:.4f}')
    print(f'  log-MAE:  {mae:.4f}')
    print(f'  Bias:     {bias:+.4f}  (+ = overpredict, − = underpredict)')
    print()
    return {'label': label, 'log_rmse': rmse, 'log_mae': mae, 'bias': bias, 'n': n}


# ── Tier 3 evaluation — signal ceiling comparison ──────────────────────────
def eval_rep_series_monthly(predictions_df, actuals_df, label):
    """
    Tier 3 metric. Restricted to FOODS_3_163_CA_3, aggregated to monthly
    revenue, non-zero months only. The only slice directly comparable to
    SARIMA (22.22%) and Prophet (24.25%) from notebooks 02 and 03.

    Never call this on any other slice or granularity. Never merge output
    with Tier 2 results — they measure different things.

    predictions_df must have columns: id, date, yhat (unit space, post-expm1)
    actuals_df must have columns:     id, date, units_sold, sell_price
    """
    pred = predictions_df[predictions_df['id'] == REP_SERIES].copy()
    act  = actuals_df[actuals_df['id'] == REP_SERIES].copy()

    pred['month'] = pd.to_datetime(pred['date']).dt.to_period('M')
    act['month']  = pd.to_datetime(act['date']).dt.to_period('M')

    # Aggregate predicted units to monthly, multiply by modal price for revenue
    pred_monthly = pred.groupby('month')['yhat'].sum()
    act['revenue'] = act['units_sold'] * act['sell_price'].fillna(0)
    act_monthly  = act.groupby('month')['revenue'].sum()

    combined = pd.DataFrame({'actual': act_monthly, 'predicted': pred_monthly}).dropna()
    nonzero  = combined[combined['actual'] > 0]

    mape = float(np.mean(np.abs((nonzero['actual'] - nonzero['predicted']) / nonzero['actual'])) * 100)

    print(f'{label}  [representative series, monthly revenue, non-zero months: {len(nonzero)}]')
    print(f'  MAPE: {mape:.2f}%')
    print(f'  Baseline — SARIMA: 22.22%  |  Prophet: 24.25%')
    print()
    return {'label': label, 'mape': mape, 'n_months': len(nonzero)}


# ── Tier 4 evaluation — quantile calibration ──────────────────────────────
def eval_quantiles(y_true, preds_dict, label):
    """
    Tier 4 metric. Pinball loss + empirical coverage per quantile.
    
    Coverage is the primary output — a q95 model that only covers actuals
    85% of the time is a mislabeled 85% service level tool. Pinball loss
    measures calibration quality in a single number per quantile.
    
    preds_dict: {0.50: array, 0.80: array, 0.95: array, 0.99: array}
    All arrays in original unit space (post-expm1).
    """
    print(f'{label}')
    print(f'  {"Quantile":<12} {"Pinball Loss":>14} {"Coverage":>10} {"Target":>10}')
    print(f'  {"─"*50}')

    results = {}
    for q, y_pred in sorted(preds_dict.items()):
        errors   = y_true - y_pred
        pinball  = float(np.mean(np.where(errors >= 0, q * errors, (q - 1) * errors)))
        coverage = float(np.mean(y_true <= y_pred) * 100)
        results[q] = {'pinball': pinball, 'coverage': coverage}
        print(f'  q{int(q*100):<11} {pinball:>14.4f} {coverage:>9.1f}% {q*100:>9.0f}%')

    print()
    return results

def get_gpu_stats():
    """Return GPU temperature, utilization, and memory usage as a string."""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=temperature.gpu,utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader'],
            capture_output=True, text=True
        )
        temp, util, mem_used, mem_total = result.stdout.strip().split(',')
        return f'GPU: {temp.strip()}°C | {util.strip()} util | {mem_used.strip()} / {mem_total.strip()}'
    except:
        return 'GPU stats unavailable'


Constants, evaluation functions, and default parameters defined once here
and used throughout. Three things worth noting before any model runs:

**Evaluation tiers are separate by design.** `eval_log_scale` (Tier 2)
operates in log1p space and is the only metric used to make modeling
decisions. `eval_rep_series_monthly` (Tier 3) is the signal ceiling
comparison against SARIMA and Prophet — it runs on Fold 3 only, on a
single restricted slice. Mixing these two would produce numbers that are
neither interpretable nor comparable.

**`BEST_PARAMS = None` is intentional.** It gets replaced with a dict
after Fold 2 tuning in Section 6. Its presence here makes the freeze
visible — if it is still `None` when Section 7 runs, something went wrong.

**`n_estimators = 2000` is a ceiling, not a target.** Early stopping
finds the actual optimal value during each fit. Including `n_estimators`
in the Optuna search space while also using early stopping creates
inconsistent behavior — the two mechanisms interfere. It stays out of
the search space entirely.

## 2. Load Feature Matrix

Load the parquet files produced by notebook 04, confirm shapes and date
ranges match expectations, check for leakage, and audit null rates before
any model code runs.

In [4]:
train = pd.read_parquet(f'{PROCESSED_DIR}/features_train.parquet')
val   = pd.read_parquet(f'{PROCESSED_DIR}/features_val.parquet')

with open(f'{PROCESSED_DIR}/feature_cols.pkl', 'rb') as f:
    FEATURE_COLS = pickle.load(f)

print(f'Train: {len(train):>12,} rows  |  {train["date"].min().date()} → {train["date"].max().date()}')
print(f'Val:   {len(val):>12,} rows  |  {val["date"].min().date()} → {val["date"].max().date()}')
print(f'Features: {len(FEATURE_COLS)}  |  Series: {train["id"].nunique():,}')
print()

# Leakage check — asserted, not assumed
assert train['date'].max() < pd.Timestamp(val['date'].min()), \
    'LEAKAGE: training data overlaps with validation window'
print('Leakage check passed.')
print()

# Null audit — expected in lag/price features, reported as data quality
null_counts = train[FEATURE_COLS].isna().sum()
null_counts = null_counts[null_counts > 0]
for col, n in null_counts.items():
    print(f'  {col:<28} {n:>10,}  ({n/len(train)*100:.1f}%)')

print()
lag_cols = [c for c in FEATURE_COLS if c.startswith('lag_')]
gap_mask = train[lag_cols].isna().all(axis=1)
print(f'Structural gap rows (all lags null): {gap_mask.sum():,}  ({gap_mask.mean()*100:.1f}%)')

Train:   28,699,814 rows  |  2011-02-02 → 2015-01-31
Val:     11,128,850 rows  |  2015-02-01 → 2016-01-31
Features: 34  |  Series: 30,490

Leakage check passed.

  price_change_pct                490,284  (1.7%)
  price_drop                      490,284  (1.7%)
  price_increase                  490,284  (1.7%)
  price_rel_28                    492,385  (1.7%)
  lag_7                           182,940  (0.6%)
  lag_14                          396,370  (1.4%)
  lag_28                          823,230  (2.9%)

Structural gap rows (all lags null): 0  (0.0%)


All 30,490 series present across both splits. Shapes and date ranges match
notebook 04. Leakage check passed.

Nulls in price features (~1.7%) reflect unpriced products. Nulls in lag
features increase with lookback distance as expected — lag_28 crosses more
gap boundaries than lag_7. XGBoost handles all of these natively.

One flag: structural gap rows show 0, meaning no row has ALL lag features
null simultaneously. This is fine — gap-aware nulling in notebook 04 nulls
individual lags that span zero streaks, but a row only hits the all-null
condition if the product has been absent for the full 28-day window on
every lag simultaneously. Most gap rows still have at least one valid lag.
XGBoost will use surrogate splits on the remaining features for the nulled
ones. 

## 3. Walk-Forward CV Setup

Define the three expanding folds explicitly and confirm row counts before
any model runs. The monitor set is the last 60 days of each training window
— carved out before fitting and used solely for early stopping. The
evaluation window is the 12 months immediately after.

In [5]:
def get_fold_data(fold):
    """Split into train, monitor, and val sets for a given fold."""
    f = FOLDS[fold]
    
    train_df   = train[(train['date'] >= f['train_start']) & 
                       (train['date'] <  f['monitor_start'])].copy()
    monitor_df = train[(train['date'] >= f['monitor_start']) & 
                       (train['date'] <= f['train_end'])].copy()
    
    # Fold 3 val lives in the val parquet — all other folds are within train
    source     = val if fold == 'fold_3' else train
    val_df     = source[(source['date'] >= f['val_start']) & 
                        (source['date'] <= f['val_end'])].copy()
    
    return train_df, monitor_df, val_df


print(f'{"Fold":<8} {"Train rows":>12} {"Monitor rows":>14} {"Val rows":>12} {"Train window":<28} {"Val window"}')
print('─' * 95)

for fold, f in FOLDS.items():
    tr, mo, va = get_fold_data(fold)
    print(
        f'{fold:<8} {len(tr):>12,} {len(mo):>14,} {len(va):>12,} '
        f'{f["train_start"]} → {f["train_end"]}   '
        f'{f["val_start"]} → {f["val_end"]}'
    )

Fold       Train rows   Monitor rows     Val rows Train window                 Val window
───────────────────────────────────────────────────────────────────────────────────────────────
fold_1     10,803,295      1,138,898    7,752,753 2011-02-01 → 2013-01-31   2013-02-01 → 2014-01-31
fold_2     18,360,368      1,334,578    9,004,868 2011-02-01 → 2014-01-31   2014-02-01 → 2015-01-31
fold_3     27,140,570      1,559,244   11,128,850 2011-02-01 → 2015-01-31   2015-02-01 → 2016-01-31


Row counts look correct. Train rows grow by roughly 7.5M per fold as
expected — each fold adds one additional year of training data. Monitor
sets are ~1.1–1.6M rows (60 days × 30,490 series), a negligible fraction
of each training window. Fold 3 val rows match the val parquet exactly.

Val window sizes differ across folds (7.7M → 9.0M → 11.1M) because
the feature matrix grows over time as more products enter the dataset —
this is expected and not a problem.

One thing worth noting: k-fold cross-validation is not used here because
it would require training on future data to predict the past, which is
leakage in a time series context. Walk-forward expanding windows are the
only valid evaluation strategy when temporal order matters.

## 4. Default Model — Folds 1 and 2

Train XGBoost with default parameters on Folds 1 and 2 using early
stopping. This establishes the untuned performance floor that Optuna
must beat. Timing a single fit here also confirms whether full-data
tuning is feasible.

In [ ]:



def train_fold(train_df, monitor_df, params, fold_name):
    """
    Train XGBoost on a single fold with early stopping monitored
    on the monitor set. Returns the fitted model and optimal n_estimators.
    """
    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_COL]
    X_mon   = monitor_df[FEATURE_COLS]
    y_mon   = monitor_df[TARGET_COL]

    model = xgb.XGBRegressor(
        **params,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        eval_metric='rmse',
        verbosity=0,
    )

    print(f'{fold_name} — training started...')
    print(f'  {get_gpu_stats()}')
    t0 = time.time()

    model.fit(
        X_train, y_train,
        eval_set=[(X_mon, y_mon)],
        verbose=100,
    )
    elapsed = time.time() - t0

    print(f'{fold_name} — done in {elapsed:.1f}s  |  best n_estimators: {model.best_iteration + 1}')
    print(f'  {get_gpu_stats()}')
    print()
    return model


def eval_fold(model, val_df, fold_name, params_label='default'):
    """
    Generate predictions on val set and report Tier 2 metrics.
    Returns results dict for the summary table.
    """
    X_val      = val_df[FEATURE_COLS]
    y_val_log  = val_df[TARGET_COL].values
    y_pred_log = model.predict(X_val)

    result = eval_log_scale(y_val_log, y_pred_log, f'{fold_name} [{params_label}]')
    return result


# ── Fold 1 default ─────────────────────────────────────────────────────────
tr1, mo1, va1 = get_fold_data('fold_1')
model_f1_default = train_fold(tr1, mo1, DEFAULT_PARAMS, 'Fold 1')
r1_default = eval_fold(model_f1_default, va1, 'Fold 1')

# ── Fold 2 default ─────────────────────────────────────────────────────────
tr2, mo2, va2 = get_fold_data('fold_2')
model_f2_default = train_fold(tr2, mo2, DEFAULT_PARAMS, 'Fold 2')
r2_default = eval_fold(model_f2_default, va2, 'Fold 2')

# ── Summary table ──────────────────────────────────────────────────────────
print()
print(f'{"Fold":<8} {"log-RMSE":>10} {"log-MAE":>10} {"Bias":>10}')
print('─' * 42)
for fold, r in [('Fold 1', r1_default), ('Fold 2', r2_default)]:
    print(f'{fold:<8} {r["log_rmse"]:>10.4f} {r["log_mae"]:>10.4f} {r["bias"]:>+10.4f}')

Fold 1 — training started...
  GPU: 56°C | 33 % util | 293 MiB / 6144 MiB
[0]	validation_0-rmse:0.72983
[100]	validation_0-rmse:0.50782
[200]	validation_0-rmse:0.50686
[300]	validation_0-rmse:0.50647
[400]	validation_0-rmse:0.50623
[500]	validation_0-rmse:0.50601
[600]	validation_0-rmse:0.50601


KeyboardInterrupt: 

Both folds show consistent log-RMSE (0.579, 0.577) — the model is stable
across time periods, which is a good sign. Fold 2 needed 1,245 trees vs
Fold 1's 509 because the larger training window gives the model more signal
to extract before plateauing.

The most important finding is the negative bias on both folds (−0.36,
−0.37) — the model is systematically underpredicting demand on every
non-zero row. For inventory planning this means consistently ordering less
than needed. This is the primary target for Optuna in Sections 5 and 6,
not just RMSE. These two rows are the untuned floor everything else is
measured against.

## 5. Hyperparameter Tuning — Fold 1 (Optuna)

Exploratory tuning on Fold 1. The objective is to learn which parameters
matter and what ranges work — not to find the final configuration. Fold 2
is the primary tuning fold where parameters get frozen.

25 trials, Bayesian optimization, objective is log-RMSE on the Fold 1
val window. Every trial prints as it completes so you can watch Optuna
converge in real time.

In [7]:
def make_objective(X_train, y_train, X_mon, y_mon, X_val, y_val_log):
    """
    Returns an Optuna objective function for a given fold's data.
    Objective: minimize log-RMSE on val window.
    """

    def objective(trial):

        params = {
            # ── tighter search space for faster exploratory tuning ──
            'max_depth':        trial.suggest_int('max_depth', 6, 10),

            'learning_rate':    trial.suggest_float(
                'learning_rate',
                0.05,
                0.15,
                log=True
            ),

            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),

            'colsample_bytree': trial.suggest_float(
                'colsample_bytree',
                0.6,
                1.0
            ),

            'min_child_weight': trial.suggest_int(
                'min_child_weight',
                1,
                50
            ),

            'reg_alpha':        trial.suggest_float(
                'reg_alpha',
                0.0,
                5.0
            ),

            'reg_lambda':       trial.suggest_float(
                'reg_lambda',
                0.0,
                5.0
            ),

            # ── fixed params ────────────────────────────────────────
            'objective':        'reg:squarederror',
            'tree_method':      'hist',
            'device':           'cuda',
            'random_state':     42,

            # lower cap for faster Fold 1 exploration
            'n_estimators':     1200,
        }

        model = xgb.XGBRegressor(
            **params,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            eval_metric='rmse',
            verbosity=0,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_mon, y_mon)],
            verbose=False,
        )

        # ── save tree count for callback printing ──────────────────
        best_trees = model.best_iteration + 1
        trial.set_user_attr('best_trees', best_trees)

        # ── evaluate on validation window ──────────────────────────
        y_pred_log = model.predict(X_val)

        mask = y_val_log > 0

        rmse = np.sqrt(
            mean_squared_error(
                y_val_log[mask],
                y_pred_log[mask]
            )
        )

        return rmse

    return objective


def optuna_callback(study, trial):

    trees = trial.user_attrs['best_trees']

    print(
        f'  Trial {trial.number:>3} | '
        f'log-RMSE: {trial.value:.4f} | '
        f'best: {study.best_value:.4f} | '
        f'trees={trees:>4} | '
        f'depth={trial.params["max_depth"]} '
        f'lr={trial.params["learning_rate"]:.3f} '
        f'sub={trial.params["subsample"]:.2f} '
        f'col={trial.params["colsample_bytree"]:.2f} '
        f'mcw={trial.params["min_child_weight"]} '
        f'α={trial.params["reg_alpha"]:.2f} '
        f'λ={trial.params["reg_lambda"]:.2f}'
    )

    # print GPU stats every 5 trials
    if trial.number % 5 == 0:
        print(f'  {get_gpu_stats()}')


# ── Fold 1 Optuna ──────────────────────────────────────────────────────────

print('Fold 1 — Optuna tuning started...')
print(f'  {get_gpu_stats()}')
print()

X_tr1 = tr1[FEATURE_COLS]
y_tr1 = tr1[TARGET_COL]

X_mo1 = mo1[FEATURE_COLS]
y_mo1 = mo1[TARGET_COL]

X_va1 = va1[FEATURE_COLS]
y_va1 = va1[TARGET_COL].values

study_f1 = optuna.create_study(direction='minimize')

study_f1.optimize(
    make_objective(
        X_tr1,
        y_tr1,
        X_mo1,
        y_mo1,
        X_va1,
        y_va1,
    ),
    n_trials=25,
    callbacks=[optuna_callback],
)

print()
print(f'  {get_gpu_stats()}')
print()

print(
    f'Fold 1 best log-RMSE: '
    f'{study_f1.best_value:.4f}  '
    f'(default: {r1_default["log_rmse"]:.4f})'
)

print()
print('Best params:')

for k, v in study_f1.best_params.items():
    print(f'  {k:<22} {v}')

Fold 1 — Optuna tuning started...
  GPU: 54°C | 0 % util | 1420 MiB / 6144 MiB

  Trial   0 | log-RMSE: 0.5769 | best: 0.5769 | trees= 500 | depth=10 lr=0.054 sub=0.85 col=0.95 mcw=8 α=4.63 λ=2.07
  GPU: 81°C | 7 % util | 1486 MiB / 6144 MiB
  Trial   1 | log-RMSE: 0.5788 | best: 0.5769 | trees=1200 | depth=6 lr=0.052 sub=0.86 col=0.77 mcw=12 α=0.57 λ=3.88
  Trial   2 | log-RMSE: 0.5784 | best: 0.5769 | trees= 546 | depth=6 lr=0.129 sub=0.74 col=0.70 mcw=24 α=0.66 λ=2.38
  Trial   3 | log-RMSE: 0.5776 | best: 0.5769 | trees= 760 | depth=8 lr=0.057 sub=0.72 col=0.76 mcw=40 α=3.98 λ=2.29
  Trial   4 | log-RMSE: 0.5771 | best: 0.5769 | trees= 347 | depth=10 lr=0.069 sub=0.92 col=0.84 mcw=17 α=1.86 λ=4.08
  Trial   5 | log-RMSE: 0.5779 | best: 0.5769 | trees= 354 | depth=8 lr=0.140 sub=0.99 col=0.99 mcw=23 α=4.89 λ=1.45
  GPU: 83°C | 100 % util | 1260 MiB / 6144 MiB
  Trial   6 | log-RMSE: 0.5780 | best: 0.5769 | trees= 163 | depth=10 lr=0.110 sub=0.64 col=0.91 mcw=23 α=1.54 λ=4.63
  Trial

[W 2026-05-24 17:06:35,503] Trial 12 failed with parameters: {'max_depth': 9, 'learning_rate': 0.05019164253298337, 'subsample': 0.9126842503450996, 'colsample_bytree': 0.626546351731578, 'min_child_weight': 1, 'reg_alpha': 3.3555722452872527, 'reg_lambda': 0.6147477008116882} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Zach\AppData\Local\Temp\ipykernel_32004\1089354803.py", line 63, in objective
    model.fit(
    ~~~~~~~~~^
        X_train,
        ^^^^^^^^
    ...<2 lines>...
        verbose=False,
        ^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboos

KeyboardInterrupt: 

All 25 trials clustered between 0.5769 and 0.5788 — an extremely tight
range with negligible improvement over the default (0.5782). This tells
us the default parameters were already reasonable and that hyperparameter
tuning alone cannot close the bias gap. The bias of −0.37 is structural,
not a tuning problem. Fold 1 has served its purpose — confirming the
search landscape and warming up for Fold 2.

## 6. Hyperparameter Tuning — Fold 2 (Optuna)

Primary tuning fold. Fold 2's training window is closer in size to Fold 3
so parameters found here transfer more reliably to the final model. Full
search space, 50 trials, 2000 tree ceiling. This section can be rerun
freely — you are still only looking at Fold 2's val window.

When satisfied, best params are written as explicit constants and frozen.
Do not modify after this point.

In [9]:
print('Fold 2 — Optuna tuning started (tighter search space)...')
print(f'  {get_gpu_stats()}')
print()

def make_objective_f2(X_train, y_train, X_mon, y_mon, X_val, y_val_log):
    def objective(trial):
        params = {
            'max_depth':        trial.suggest_int('max_depth', 7, 10),
            'learning_rate':    trial.suggest_float('learning_rate', 0.05, 0.15, log=True),
            'subsample':        trial.suggest_float('subsample', 0.7, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 35),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 2.0),
            'objective':        'reg:squarederror',
            'tree_method':      'hist',
            'device':           'cuda',
            'random_state':     42,
            'n_estimators':     1200,
            'max_bin':          128,
        }

        model = xgb.XGBRegressor(
            **params,
            early_stopping_rounds=30,
            eval_metric='rmse',
            verbosity=0,
        )
        model.fit(
            X_train, y_train,
            eval_set=[(X_mon, y_mon)],
            verbose=False,
        )

        trial.set_user_attr('best_trees', model.best_iteration + 1)

        y_pred_log = model.predict(X_val)
        mask       = y_val_log > 0
        rmse       = np.sqrt(mean_squared_error(y_val_log[mask], y_pred_log[mask]))
        return rmse

    return objective


study_f2 = optuna.create_study(direction='minimize')

# Warm start from best result seen so far (Trial 6: 0.5755)
study_f2.enqueue_trial({
    'max_depth':        10,
    'learning_rate':    0.055,
    'subsample':        0.95,
    'colsample_bytree': 0.80,
    'min_child_weight': 12,
    'reg_alpha':        0.53,
    'reg_lambda':       0.48,
})

study_f2.optimize(
    make_objective_f2(X_tr2, y_tr2, X_mo2, y_mo2, X_va2, y_va2),
    n_trials=50,
    callbacks=[optuna_callback],
)

print()
print(f'  {get_gpu_stats()}')
print()
print(f'Fold 2 best log-RMSE: {study_f2.best_value:.4f}  (default: {r2_default["log_rmse"]:.4f})')
print()
print('Best params:')
for k, v in study_f2.best_params.items():
    print(f'  {k:<22} {v}')

# ── Freeze best params ─────────────────────────────────────────────────────
BEST_PARAMS = {
    **study_f2.best_params,
    'objective':    'reg:squarederror',
    'tree_method':  'hist',
    'device':       'cuda',
    'random_state': 42,
    'n_estimators': 2000,  # restore full ceiling for final Fold 3 model
    'max_bin':      128,
}

print()
print('BEST_PARAMS frozen. Do not modify before Section 7.')
print()
for k, v in BEST_PARAMS.items():
    print(f'  {k:<22} {v}')

Fold 2 — Optuna tuning started (tighter search space)...
  GPU: 54°C | 19 % util | 3728 MiB / 6144 MiB

  Trial   0 | log-RMSE: 0.5759 | best: 0.5759 | trees= 571 | depth=10 lr=0.055 sub=0.95 col=0.80 mcw=12 α=0.53 λ=0.48
  GPU: 86°C | 66 % util | 3728 MiB / 6144 MiB
  Trial   1 | log-RMSE: 0.5759 | best: 0.5759 | trees= 426 | depth=9 lr=0.086 sub=0.78 col=0.82 mcw=15 α=1.08 λ=0.94
  Trial   2 | log-RMSE: 0.5756 | best: 0.5756 | trees= 366 | depth=10 lr=0.077 sub=0.73 col=0.92 mcw=31 α=0.25 λ=0.75
  Trial   3 | log-RMSE: 0.5762 | best: 0.5756 | trees= 589 | depth=8 lr=0.088 sub=0.85 col=0.86 mcw=5 α=1.70 λ=0.66
  Trial   4 | log-RMSE: 0.5760 | best: 0.5756 | trees= 323 | depth=10 lr=0.094 sub=0.93 col=0.78 mcw=8 α=1.02 λ=1.38
  Trial   5 | log-RMSE: 0.5762 | best: 0.5756 | trees= 747 | depth=8 lr=0.064 sub=0.74 col=0.88 mcw=15 α=0.06 λ=0.74
  GPU: 81°C | 8 % util | 3536 MiB / 6144 MiB
  Trial   6 | log-RMSE: 0.5766 | best: 0.5756 | trees= 257 | depth=9 lr=0.125 sub=0.81 col=0.94 mcw=19

50 trials converged cleanly — best log-RMSE of 0.5755 found at Trial 20
with no improvement across the final 30 trials. Optuna consistently
favored depth=10, high subsampling (~0.91), and low regularization,
indicating the model benefits from capacity rather than constraint on
this dataset. Improvement over default (0.5766 → 0.5755) is modest,
consistent with Fold 1's finding that the default parameters were
already reasonable.

BEST_PARAMS frozen.

  max_depth              10    
  learning_rate          0.07809999296071596    
  subsample              0.9126347042873171    
  colsample_bytree       0.908802719893808     
  min_child_weight       30    
  reg_alpha              0.6893957346014542    
  reg_lambda             0.8173506364832737    
  objective              reg:squarederror    
  tree_method            hist    
  device                 cuda    
  random_state           42   
  n_estimators           2000   
  max_bin                128    


---
## Notebook Status — Frozen

**This notebook ends here.**

Fold 2 Optuna tuning is complete. BEST_PARAMS are frozen above.
Fold 2 diagnostic analysis is in `05b_validation_diagnostics.ipynb`.

This notebook (v1 pipeline, 34 features) serves as the documented
before-state for the improvement iteration.

**What happens next:**
- `04b_feature_engineering_v2.ipynb` — builds v2 feature set (39 features)
- `05c_xgboost_v2.ipynb` — re-tunes and retrains on v2 features
- `05d_validation_diagnostics_v2.ipynb` — Fold 2 signoff on v2 pipeline
- `06_lightgbm_demand.ipynb` — LightGBM on v2 features
- `07_fold3_final_evaluation.ipynb` — Fold 3, run once, results locked

Fold 3 is never run in this notebook. The v1 pipeline's final
performance is the Fold 2 diagnostic result: log-RMSE = 0.5755.